# Daily Practice — 2026-11-25 — ML Testing & MLOps: Building a Pre-Deployment Test Suite

**Dataset:** [Titanic passenger list](https://github.com/mwaskom/seaborn-data) — 891 real passenger
records from the RMS Titanic (class, sex, age, fare, family size, port of embarkation, survival),
loaded from the `mwaskom/seaborn-data` GitHub repo (the same real-world dataset seaborn ships as
`sns.load_dataset("titanic")`).

## Problem statement

A teammate has opened a pull request that trains a `RandomForestClassifier` to predict Titanic
passenger survival, to replace a hand-written "women and children first" heuristic the team has
been using as a rule-based fallback. Your job is **not** to build the best model — a reasonable one
already exists below. Your job is to build the automated test suite that decides whether *any* future
version of this model is safe to merge, the kind of suite that would run in CI on every pull request
that touches the training pipeline.

You will write four checks, each catching a different class of real production incident:

1. **`validate_schema`** — a data-contract validator that fails fast on schema drift, disallowed
   categories, out-of-range values, and excessive missingness, *before* anything reaches training.
2. **`quality_gate`** — a check that the trained model beats two baselines (a dumb majority-class
   predictor, and the non-ML "women and children first" heuristic it's meant to replace) by a
   required margin, so a model that merely matches common sense can't sneak through as "high accuracy."
3. **`check_reproducibility`** — a check that distinguishes genuine run-to-run determinism (same seed
   in, identical predictions out) from acceptable training-noise variance (different seeds, small
   accuracy spread) from a pipeline that is unacceptably unstable.
4. **`regression_test`** — a prediction-level check against a frozen "golden" prediction set, so a
   change that keeps (or even slightly improves) aggregate accuracy, but quietly flips who the model
   predicts would survive, still gets caught.

**What you should produce:** all four functions implemented and exercised against both a clean and a
deliberately corrupted version of the data / model, plus a short write-up (at the bottom) on which
check would catch which kind of production bug — and which wouldn't.

## Setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", 20)

## Load the data

We keep only the columns relevant to a from-scratch model and drop the columns seaborn derives
*from* the target or from each other (`alive` is a string copy of `survived`; `class`/`who`/
`adult_male`/`alone`/`embark_town` are redundant encodings of `pclass`/`sex`/`age`/`sibsp`+`parch`/
`embarked`). `age` is missing for about 20% of passengers and `embarked` for 2 of 891 — both are
genuine missingness in the original 1912 passenger records, not injected for this exercise.

In [ ]:
TITANIC_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

df = pd.read_csv(TITANIC_URL)
df = df.drop(columns=["deck", "alive", "class", "who", "adult_male", "alone", "embark_town"])

print(df.shape)
print(df.isna().mean().round(4))
df.head()

## Part 1 — Data contract validation

Below is `CONTRACT`: a hand-written data contract describing what every incoming batch of this data
is *supposed* to look like — one entry per column, with an expected dtype, an allowed set of
categories and/or a numeric range (where meaningful), and a maximum tolerable null rate. This is
exactly the kind of thing a real pipeline would check before a batch is even handed to feature
engineering, let alone training — cheap, fast, and deterministic.

Your task: write `validate_schema(frame, contract)` so it enforces this contract.

In [ ]:
CONTRACT = {
    "survived": {"dtype": "int64",   "allowed": {0, 1},            "max_null_rate": 0.0},
    "pclass":   {"dtype": "int64",   "allowed": {1, 2, 3},         "max_null_rate": 0.0},
    "sex":      {"dtype": "object",  "allowed": {"male", "female"},"max_null_rate": 0.0},
    "age":      {"dtype": "float64", "min": 0, "max": 100,         "max_null_rate": 0.3},
    "sibsp":    {"dtype": "int64",   "min": 0, "max": 10,          "max_null_rate": 0.0},
    "parch":    {"dtype": "int64",   "min": 0, "max": 10,          "max_null_rate": 0.0},
    "fare":     {"dtype": "float64", "min": 0, "max": 600,         "max_null_rate": 0.0},
    "embarked": {"dtype": "object",  "allowed": {"S", "C", "Q"},   "max_null_rate": 0.02},
}

## Your task

Fill in the TODOs below. Signatures are given — don't change them, the solution cells (and your
own later cells) assume these names.

In [ ]:
def validate_schema(frame, contract):
    """Check `frame` against `contract` and return a list of human-readable violation strings
    (empty list = passes). For each column in `contract`, check in this order and, if the column
    is present, ALL of the following (a single column can produce more than one violation):

      - MISSING_COLUMN : the column from `contract` is absent from `frame`           -> skip the
        rest of the checks for that column if this fires.
      - DTYPE           : frame[col].dtype does not match contract rule's "dtype"
                          ("object" should match pandas object/str/string dtypes)
      - NULL_RATE       : frame[col].isna().mean() exceeds rule["max_null_rate"] (if present)
      - BAD_CATEGORY    : frame[col] (ignoring NaNs) contains values outside rule["allowed"]
                          (if present)
      - OUT_OF_RANGE    : frame[col] (ignoring NaNs) contains values below rule["min"] or above
                          rule["max"] (if either is present)

    Each violation string should start with its code name (e.g. "NULL_RATE: ...") so a caller can
    programmatically group/count violations by type.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# TODO:
# 1. Run validate_schema(df, CONTRACT) on the clean data and confirm it returns an empty list.
# 2. Build a corrupted copy `bad` of df with these four deliberate issues:
#       - inflate `age`'s null rate well past the 30% ceiling
#       - set a few `sex` values to the invalid code "M" (instead of "male")
#       - set one `fare` value to a negative number
#       - set one `pclass` value to 0 (not a real cabin class)
# 3. Run validate_schema(bad, CONTRACT) and print each violation on its own line.


## Part 2 — Train/test split and feature prep

In [ ]:
def prepare_features(frame, age_median):
    """Return (X, y) ready for sklearn.

    - Fill missing `age` with the given `age_median` (NOT recomputed from `frame` itself — the
      median must come from the TRAINING split only, to avoid leaking test-set statistics).
    - Fill missing `embarked` with the mode "S".
    - One-hot encode `sex`, `embarked`, and `pclass` with drop_first=True.
    - Keep columns: pclass, sex, age, sibsp, parch, fare, embarked (pre-encoding).
    - y is the `survived` column as a numpy array.
    """
    # TODO
    raise NotImplementedError


# TODO:
# 1. Split `df` into train_df / test_df (test_size=0.2, stratify=df["survived"],
#    random_state=RANDOM_STATE).
# 2. Compute age_median from train_df ONLY.
# 3. Build X_train, y_train and X_test, y_test with prepare_features.
# 4. Reindex X_test's columns to match X_train's (fill_value=0) in case one-hot encoding produced
#    a different column set on each side.

## Part 3 — Train the model and two baselines

In [ ]:
def train_model(X_train, y_train, random_state=RANDOM_STATE, max_depth=5):
    """Fit and return a RandomForestClassifier(n_estimators=300, max_depth=max_depth,
    random_state=random_state, n_jobs=-1)."""
    # TODO
    raise NotImplementedError


def heuristic_baseline_predict(frame_raw, age_median):
    """The non-ML baseline this model is meant to replace: predict survived=1 for every passenger
    who is female OR younger than 16 (using age_median to fill missing ages first), else 0.
    Return a numpy array of 0/1 predictions aligned with frame_raw's row order."""
    # TODO
    raise NotImplementedError

In [ ]:
# TODO:
# 1. Train the model on (X_train, y_train).
# 2. Compute model_acc on (X_test, y_test).
# 3. Compute majority_acc: accuracy of always predicting train_df["survived"].mode()[0].
# 4. Compute heuristic_acc: accuracy of heuristic_baseline_predict(test_df, age_median).
# 5. Print all three accuracies.


## Part 4 — Quality gate: beat the baselines by a margin

"High accuracy" is meaningless on its own — a model needs to beat *something*. Here, that something
is two baselines: a baseline that requires no data at all (majority class), and a baseline that
requires no ML at all (the heuristic this model is supposed to replace).

In [ ]:
def quality_gate(model_acc, baseline_accs, min_margin=0.03):
    """`baseline_accs` is a dict like {"majority": 0.61, "heuristic": 0.78}.
    Return a dict {baseline_name: {"baseline_acc": ..., "margin": ..., "passed": bool}} where
    "margin" = model_acc - baseline_acc and "passed" = margin >= min_margin, for every entry in
    baseline_accs."""
    # TODO
    raise NotImplementedError


# TODO: call quality_gate(model_acc, {"majority": majority_acc, "heuristic": heuristic_acc}) and
# print the result.

## Part 5 — Reproducibility check

Two different failure modes hide behind "the model isn't reproducible": (a) the *exact same* training
run, same seed, same data, gives different predictions on different executions — a real determinism
bug (e.g. an unseeded shuffle somewhere in the pipeline); versus (b) *different* seeds give
somewhat different models, which is normal — the question is whether the resulting accuracy spread is
small enough to trust any single run's reported number.

In [ ]:
def check_reproducibility(X_train, y_train, X_test, y_test, seeds=range(10), max_std=0.02):
    """
    - Train the model twice with the SAME seed (seeds[0]) and check the two prediction arrays are
      byte-for-byte identical (np.array_equal). Store this bool as "same_seed_deterministic".
    - Train the model once per seed in `seeds` (different seeds) and collect test accuracy for
      each. Compute the standard deviation of those accuracies as "cross_seed_accuracy_std".
    - Return {"same_seed_deterministic": bool, "cross_seed_accuracy_std": float,
              "cross_seed_accuracies": list[float], "stable": cross_seed_accuracy_std <= max_std}.
    """
    # TODO
    raise NotImplementedError


# TODO: call check_reproducibility(X_train, y_train, X_test, y_test) and print the result.

## Part 6 — Golden-set regression test

Freeze the predictions of a known-good ("golden") model version on a fixed test set. Every future
candidate model gets scored not just on aggregate accuracy, but on how many *individual*
predictions it flips relative to the golden set. A refactor that is accuracy-neutral (or even an
improvement) can still be a behavioral regression if it changes who the model thinks will live or
die — and aggregate accuracy alone will never tell you that.

In [ ]:
def regression_test(golden_preds, new_preds, max_flip_rate=0.05):
    """Compare two 0/1 prediction arrays of equal length. Return
    {"flip_rate": float, "n_flipped": int, "passed": flip_rate <= max_flip_rate}."""
    # TODO
    raise NotImplementedError

In [ ]:
# TODO:
# 1. Train the "golden" model: train_model(X_train, y_train, max_depth=5) (the same call as Part 3).
#    Freeze its predictions on X_test as `golden_preds`, and its accuracy as `golden_acc`.
# 2. Simulate a teammate's "harmless-looking" refactor PR: train a second model with
#    train_model(X_train, y_train, max_depth=None) (same seed, depth cap accidentally removed).
#    Get its predictions as `candidate_preds` and accuracy as `candidate_acc`.
# 3. Print golden_acc and candidate_acc side by side.
# 4. Call regression_test(golden_preds, candidate_preds) and print the result.
# 5. Also run this candidate model back through quality_gate() against the same baselines, and
#    compare what quality_gate() says is fine against what regression_test() says is fine.


### Write-up

TODO: in 3–5 sentences, answer: which of the four checks (`validate_schema`, `quality_gate`,
`check_reproducibility`, `regression_test`) would catch each of these three real incidents, and
which would completely miss it?

  (a) An upstream data export starts sending `embarked` as full city names ("Southampton") instead
      of the letter codes ("S") the pipeline expects.
  (b) A teammate's "simplify the code" refactor PR accidentally removes the `max_depth` cap on the
      forest, and CI's accuracy check still goes green.
  (c) The training job starts reading from a data loader that shuffles row order differently on
      every run, with no fixed seed anywhere in the shuffle.

Then, using a concrete number from Part 6, say why a CI pipeline that gates merges on aggregate
test accuracy *alone* is not sufficient to catch behavioral regressions in a model.

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# Part 1 — validate_schema
def validate_schema(frame, contract):
    violations = []
    for col, rules in contract.items():
        if col not in frame.columns:
            violations.append(f"MISSING_COLUMN: '{col}' not found")
            continue
        series = frame[col]
        actual_dtype = str(series.dtype)
        expected_dtype = rules.get("dtype")
        if expected_dtype == "object" and actual_dtype not in ("object", "str", "string"):
            violations.append(f"DTYPE: '{col}' expected object/str-like, got {actual_dtype}")
        elif expected_dtype not in (None, "object") and actual_dtype != expected_dtype:
            violations.append(f"DTYPE: '{col}' expected {expected_dtype}, got {actual_dtype}")

        max_null_rate = rules.get("max_null_rate")
        if max_null_rate is not None:
            null_rate = series.isna().mean()
            if null_rate > max_null_rate:
                violations.append(
                    f"NULL_RATE: '{col}' null rate {null_rate:.1%} exceeds max {max_null_rate:.0%}"
                )

        allowed = rules.get("allowed")
        if allowed is not None:
            bad_values = set(series.dropna().unique()) - allowed
            if bad_values:
                violations.append(f"BAD_CATEGORY: '{col}' has disallowed values {sorted(map(str, bad_values))}")

        lo, hi = rules.get("min"), rules.get("max")
        if lo is not None or hi is not None:
            numeric = pd.to_numeric(series, errors="coerce").dropna()
            n_bad = 0
            if lo is not None:
                n_bad += int((numeric < lo).sum())
            if hi is not None:
                n_bad += int((numeric > hi).sum())
            if n_bad > 0:
                violations.append(f"OUT_OF_RANGE: '{col}' has {n_bad} value(s) outside [{lo}, {hi}]")
    return violations

print("clean:", validate_schema(df, CONTRACT))

bad = df.copy()
bad.loc[bad.index % 3 == 0, "age"] = np.nan   # pushes age null rate well past 30%
bad.loc[0:5, "sex"] = "M"                     # invalid category
bad.loc[10, "fare"] = -5.0                    # out of range
bad.loc[20, "pclass"] = 0                     # invalid category
for v in validate_schema(bad, CONTRACT):
    print(" -", v)

# Part 2 — split and features
def prepare_features(frame, age_median):
    frame = frame.copy()
    frame["age"] = frame["age"].fillna(age_median)
    frame["embarked"] = frame["embarked"].fillna("S")
    X = pd.get_dummies(
        frame[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]],
        columns=["sex", "embarked", "pclass"], drop_first=True,
    )
    y = frame["survived"].values
    return X, y

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["survived"]
)
age_median = train_df["age"].median()
X_train, y_train = prepare_features(train_df, age_median)
X_test, y_test = prepare_features(test_df, age_median)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Part 3 — model + baselines
def train_model(X_train, y_train, random_state=RANDOM_STATE, max_depth=5):
    model = RandomForestClassifier(
        n_estimators=300, max_depth=max_depth, random_state=random_state, n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model

def heuristic_baseline_predict(frame_raw, age_median):
    age = frame_raw["age"].fillna(age_median)
    return ((frame_raw["sex"] == "female") | (age < 16)).astype(int).values

model = train_model(X_train, y_train)
model_preds = model.predict(X_test)
model_acc = accuracy_score(y_test, model_preds)

majority_class = train_df["survived"].mode()[0]
majority_acc = accuracy_score(y_test, np.full_like(y_test, majority_class))

heuristic_preds = heuristic_baseline_predict(test_df, age_median)
heuristic_acc = accuracy_score(y_test, heuristic_preds)

print(f"model acc={model_acc:.4f}  majority acc={majority_acc:.4f}  heuristic acc={heuristic_acc:.4f}")

# Part 4 — quality gate
def quality_gate(model_acc, baseline_accs, min_margin=0.03):
    result = {}
    for name, acc in baseline_accs.items():
        margin = model_acc - acc
        result[name] = {"baseline_acc": acc, "margin": margin, "passed": margin >= min_margin}
    return result

print(quality_gate(model_acc, {"majority": majority_acc, "heuristic": heuristic_acc}))

# Part 5 — reproducibility
def check_reproducibility(X_train, y_train, X_test, y_test, seeds=range(10), max_std=0.02):
    seed0 = list(seeds)[0]
    preds_a = train_model(X_train, y_train, random_state=seed0).predict(X_test)
    preds_b = train_model(X_train, y_train, random_state=seed0).predict(X_test)
    same_seed_deterministic = bool(np.array_equal(preds_a, preds_b))

    accs = []
    for s in seeds:
        m = train_model(X_train, y_train, random_state=s)
        accs.append(accuracy_score(y_test, m.predict(X_test)))
    std = float(np.std(accs))
    return {
        "same_seed_deterministic": same_seed_deterministic,
        "cross_seed_accuracy_std": std,
        "cross_seed_accuracies": [round(a, 4) for a in accs],
        "stable": std <= max_std,
    }

repro = check_reproducibility(X_train, y_train, X_test, y_test)
print(repro)

# Part 6 — golden-set regression test
def regression_test(golden_preds, new_preds, max_flip_rate=0.05):
    flip_rate = float(np.mean(golden_preds != new_preds))
    return {"flip_rate": flip_rate, "n_flipped": int(np.sum(golden_preds != new_preds)),
            "passed": flip_rate <= max_flip_rate}

golden_model = train_model(X_train, y_train, max_depth=5)
golden_preds = golden_model.predict(X_test)
golden_acc = accuracy_score(y_test, golden_preds)

candidate_model = train_model(X_train, y_train, max_depth=None)  # "harmless" refactor
candidate_preds = candidate_model.predict(X_test)
candidate_acc = accuracy_score(y_test, candidate_preds)

print(f"golden acc={golden_acc:.4f}  candidate acc={candidate_acc:.4f}")
print(regression_test(golden_preds, candidate_preds))
print(quality_gate(candidate_acc, {"majority": majority_acc, "heuristic": heuristic_acc}))
```

**Typical output**

```
clean: []
 - BAD_CATEGORY: 'pclass' has disallowed values ['0']
 - BAD_CATEGORY: 'sex' has disallowed values ['M']
 - NULL_RATE: 'age' null rate 47.3% exceeds max 30%
 - OUT_OF_RANGE: 'fare' has 1 value(s) outside [0, 600]

model acc=0.8156  majority acc=0.6145  heuristic acc=0.7821

{'majority': {'baseline_acc': 0.6145251396648045, 'margin': 0.20111726384575825, 'passed': True},
 'heuristic': {'baseline_acc': 0.7821229050279329, 'margin': 0.03351955307262576, 'passed': True}}

{'same_seed_deterministic': True, 'cross_seed_accuracy_std': 0.0037,
 'cross_seed_accuracies': [0.8156, 0.8156, 0.8156, 0.81, 0.81, 0.8156, 0.8156, 0.8045, 0.81, 0.8156],
 'stable': True}

golden acc=0.8156  candidate acc=0.8212
{'flip_rate': 0.1061, 'n_flipped': 19, 'passed': False}
{'majority': {'baseline_acc': 0.6145..., 'margin': 0.2067..., 'passed': True},
 'heuristic': {'baseline_acc': 0.7821..., 'margin': 0.0391..., 'passed': True}}
```

**Explanation**

The clean dataset passes every contract check with zero violations, and the corrupted copy is
flagged on exactly the four columns it was corrupted on — no false positives, no false negatives.
This is the test that should run first, before a single model is trained: it's the only one of the
four that catches bad data at the source rather than bad *behavior* downstream.

The quality gate passes comfortably against the majority-class baseline (+20 points) but only
*barely* against the heuristic baseline it's meant to replace: +3.4 points against a 3-point minimum
margin. That's a legitimate and useful finding, not a bug — it says a random forest on these seven
features buys you only a little over "women and children first," and a reviewer should know that
before merging, not discover it later from a confused stakeholder asking why the fancy ML model
barely beats common sense.

The reproducibility check shows both failure modes are absent here: the exact same seed gives
byte-identical predictions (no hidden non-determinism in the training code), and accuracy across
ten different seeds has a standard deviation of under half a percentage point — this training setup
is boring and stable, which is exactly what you want to see.

Part 6 is the one that should change how you think about CI gates. The "harmless refactor" that
drops `max_depth` from 5 to `None` doesn't fail `quality_gate` — accuracy actually *improves* slightly,
from 81.6% to 82.1%, and the margin against both baselines gets slightly wider. An accuracy-only CI
check would wave this PR straight through. But `regression_test` shows it flips **19 of 179**
individual test-set predictions (a 10.6% flip rate) relative to the golden model — more than double
the 5% tolerance. Those 19 people are predicted to survive when the golden model said they wouldn't,
or vice versa, even though the headline number went *up*. For a model whose predictions feed into
any downstream decision (resource allocation, triage, risk scoring), "the aggregate metric looks the
same or better" is not the same guarantee as "the model behaves the same way," and only a
prediction-level regression test against a frozen reference set can tell the two apart.

</details>